In [35]:
import pandas as pd

In [36]:
URL = "https://raw.githubusercontent.com/patcg-individual-drafts/topics/main/taxonomy_v2.md"

df = pd.read_csv(
    URL,
    sep="|",
    skipinitialspace=True
)

# Clean Markdown table formatting
df.columns = [c.strip() for c in df.columns]
df = df.iloc[1:, :]


In [37]:
df.head()

,Unnamed: 0,ID,Topic,Unnamed: 3
1,NaN,1,/Arts & Entertainment ...,NaN
2,NaN,350,/Arts & Entertainment/Celebrities & Entertainm...,NaN
3,NaN,351,/Arts & Entertainment/Comics & Animation ...,NaN
4,NaN,352,/Arts & Entertainment/Events & Listings ...,NaN
5,NaN,353,"/Arts & Entertainment/Events & Listings/Bars, ...",NaN


In [38]:

df["ID"] = (
    df["ID"]
    .astype(str)
    .str.strip()
    .astype(int)
)

df["Topic"] = (
    df["Topic"]
    .astype(str)
    .str.strip()
)

df = df[['ID', 'Topic']]

print(df.head())
print(f"Total topics: {len(df)}")

    ID                                              Topic
1    1                              /Arts & Entertainment
2  350  /Arts & Entertainment/Celebrities & Entertainm...
3  351           /Arts & Entertainment/Comics & Animation
4  352            /Arts & Entertainment/Events & Listings
5  353  /Arts & Entertainment/Events & Listings/Bars, ...
Total topics: 469


In [39]:
# Separate Topic by delimiter '/'
topic_expanded = df['Topic'].str.split('/', expand=True)
# The first column is empty because of the leading slash
topic_expanded = topic_expanded.iloc[:, 1:]
topic_expanded.columns = [f'Topic_Level_{i+1}' for i in range(topic_expanded.shape[1])]
df = pd.concat([df, topic_expanded], axis=1)
df.head()

,ID,Topic,Topic_Level_1,Topic_Level_2,Topic_Level_3,Topic_Level_4,Topic_Level_5
1,1,/Arts & Entertainment,Arts & Entertainment,None,None,None,None
2,350,/Arts & Entertainment/Celebrities & Entertainm...,Arts & Entertainment,Celebrities & Entertainment News,None,None,None
3,351,/Arts & Entertainment/Comics & Animation,Arts & Entertainment,Comics & Animation,None,None,None
4,352,/Arts & Entertainment/Events & Listings,Arts & Entertainment,Events & Listings,None,None,None
5,353,"/Arts & Entertainment/Events & Listings/Bars, ...",Arts & Entertainment,Events & Listings,"Bars, Clubs & Nightlife",None,None


In [40]:
# Merge Topic_Level_2 through Topic_Level_5 into sub_topic
df['sub_topic'] = df['Topic_Level_2'].copy()
for i in range(3, 6):
    col = f'Topic_Level_{i}'
    if col in df.columns:
        mask = df[col].notna() & (df[col] != '')
        df.loc[mask, 'sub_topic'] = df.loc[mask, 'sub_topic'] + ' + ' + df.loc[mask, col]

df[['Topic', 'sub_topic']].head(10)

,Topic,sub_topic
1,/Arts & Entertainment,None
2,/Arts & Entertainment/Celebrities & Entertainm...,Celebrities & Entertainment News
3,/Arts & Entertainment/Comics & Animation,Comics & Animation
4,/Arts & Entertainment/Events & Listings,Events & Listings
5,"/Arts & Entertainment/Events & Listings/Bars, ...","Events & Listings + Bars, Clubs & Nightlife"
6,/Arts & Entertainment/Events & Listings/Concer...,Events & Listings + Concerts & Music Festivals
7,/Arts & Entertainment/Events & Listings/Event ...,Events & Listings + Event Ticket Sales
8,/Arts & Entertainment/Events & Listings/Expos ...,Events & Listings + Expos & Conventions
9,/Arts & Entertainment/Events & Listings/Film F...,Events & Listings + Film Festivals
10,/Arts & Entertainment/Events & Listings/Food &...,Events & Listings + Food & Beverage Events


In [41]:
# Remove rows where Topic_Level_1 matches specific categories
topics_to_remove = [
    "Books & Literature",
    "Hobbies & Leisure",
    "People & Society",
    "Pets & Animals",
    "News",
    "Law & Government",
    "Online Communities"
]

df = df[~df['Topic_Level_1'].isin(topics_to_remove)]
print(f"Remaining topics: {len(df)}")
df['Topic_Level_1'].value_counts()

Remaining topics: 432


Topic_Level_1
Home & Garden              60
Arts & Entertainment       49
Shopping                   45
Computers & Electronics    35
Jobs & Education           34
Autos & Vehicles           34
Sports                     29
Beauty & Fitness           28
Finance                    22
Business & Industrial      21
Food & Drink               19
Travel & Transportation    18
Internet & Telecom         16
Games                      13
Real Estate                 9
Name: count, dtype: int64

In [42]:
# Export taxonomy for bulk ad generation
import os
out_dir = '../data/processed/lmarena'
os.makedirs(out_dir, exist_ok=True)
df[['Topic_Level_1', 'sub_topic']].to_csv(os.path.join(out_dir, 'taxonomy_subtopics.csv'), index=False)
print('Saved taxonomy_subtopics.csv')

Saved taxonomy_subtopics.csv
